# Hand Rehab Coach — full ML training & evaluation notebook
Runtime → Change runtime type → **T4 GPU**, then **Run all** (about 40–70 min on `standard`; use `quick` for a 15-min pass).

What this notebook trains and measures (all with **subject-wise cross-validation** — a subject is never in train and test):

| # | Method | Why |
|---|---|---|
| 1 | Mean / Ridge / XGBoost on FFT-tremor, SPARC smoothness, jerk, rhythm features | honest classical baseline |
| 2 | BiGRU | recurrent baseline |
| 3 | Spatial Transformer (bone-tree attention) + **Mamba selective state-space** temporal blocks | the teacher |
| 4 | Masked-joint **self-supervised** pretraining, + **contrastive (NT-Xent)** pretraining | ablation ladder |
| 5 | **Conditional VAE** generative augmentation | ablation |
| 6 | **Domain-adversarial** training (only if you upload real recordings) | sim-to-real |
| 7 | **One-class VAE** scoring trained on good form only | label-free quality score |
| 8 | Heteroscedastic head + **split-conformal** intervals | calibrated uncertainty |
| 9 | **Integrated Gradients** + deletion faithfulness test | which finger/joint caused the cue |
| 10 | **Few-shot low-rank personalization** | adapt to a new patient |
| 11 | **Distillation** to a 1D-CNN student → **ONNX int8** | runs inside the Quest browser |

⚠️ Labels come from a synthetic kinematic rubric unless you add real recordings. Report synthetic numbers as synthetic.

In [ ]:
import os, sys, subprocess, zipfile, glob
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/handrehab_ckpt'   # checkpoints survive disconnects; re-running resumes
else:
    CKPT = './ckpt'
os.makedirs(CKPT, exist_ok=True)
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 0. Get the code
Upload `handrehab.zip` (the project zip) with the file panel on the left, or run the upload cell.

In [ ]:
if not os.path.exists('ml/train.py'):
    zs = glob.glob('*.zip')
    if not zs and IN_COLAB:
        from google.colab import files; up = files.upload(); zs = list(up)
    zipfile.ZipFile(zs[0]).extractall('.')
    if not os.path.exists('ml/train.py'):   # zip may contain a top folder
        top = [d for d in os.listdir('.') if os.path.exists(f'{d}/ml/train.py')][0]; os.chdir(top)
os.chdir('ml'); print(os.getcwd())
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])

In [ ]:
PRESET = 'standard'      # 'quick' | 'standard'
REAL_DIR = '../recordings'   # optional: JSON takes recorded in the app with ?mode=record (enables the domain-adversarial row)
def run(cmd):
    cmd = f'{sys.executable} -u train.py {cmd} --ckpt {CKPT} --preset {PRESET} --real {REAL_DIR}'
    print('$', cmd); subprocess.run(cmd, shell=True, check=True)

## 1. Baselines — classical signal features
ROM, tremor-band (4–12 Hz) power, SPARC spectral smoothness, normalised jerk, rhythm CV … fed to Ridge and XGBoost.

In [ ]:
run('baselines')

## 2. Self-supervised pretraining (no labels)
Mask 30 % of joint×frame tokens and reconstruct them; optionally add a SimCLR objective over augmented views (time-warp, rotation, mirror, noise).

In [ ]:
run('pretrain --ssl recon')

In [ ]:
run('pretrain --ssl both')

## 3. Cross-validation ablation ladder
Each row = same folds, same data. Only the method changes.

In [ ]:
run('cv --tag lstm    --arch lstm   --init scratch')

In [ ]:
run('cv --tag scratch --arch mamba  --init scratch')

In [ ]:
run('cv --tag recon   --arch mamba  --init recon')

In [ ]:
run('cv --tag both    --arch mamba  --init both')

### 3b. + generative augmentation (conditional VAE trained inside each training fold only)

In [ ]:
run('cv --tag both_gen --arch mamba --init both --gen')

### 3c. + domain-adversarial (only meaningful if `recordings/` contains real takes)

In [ ]:
if os.path.isdir(REAL_DIR) and glob.glob(REAL_DIR+'/*.json'): run('cv --tag both_dann --arch mamba --init both --dann')
else: print('No real recordings found - skipping (this is expected on a first run).')

## 4. Final teacher, one-class VAE, uncertainty, explanations, personalization

In [ ]:
run('final --tag both --arch mamba --init both')

In [ ]:
run('vae')

In [ ]:
run('conformal --tag both')

In [ ]:
run('explain --tag both')

In [ ]:
run('personalize --tag both')

## 5. Distil to the headset model and export ONNX (int8)

In [ ]:
run('distill --tag both --from fold0')   # honest held-out comparison teacher vs student

In [ ]:
run('distill --tag both --from final')   # model you ship

In [ ]:
run('export --tag both')

## 6. Results

In [ ]:
run('report')
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{CKPT}/report/summary.md').read()))
for f in ('ladder','joint_importance','personalization'):
    p=f'{CKPT}/report/{f}.png'
    if os.path.exists(p): display(Image(p))

## 7. Download the files the app needs
Copy `student.int8.onnx` and `calibration.json` into the app's `public/models/`, then redeploy.

In [ ]:
out = 'handrehab_results.zip'
with zipfile.ZipFile(out,'w') as z:
    for f in ['student.int8.onnx','student.onnx','calibration.json','model_card.json','student_eval.json']:
        if os.path.exists(f'{CKPT}/{f}'): z.write(f'{CKPT}/{f}', f)
    for f in glob.glob(f'{CKPT}/*.json')+glob.glob(f'{CKPT}/report/*'): z.write(f, 'metrics/'+os.path.basename(f))
if IN_COLAB:
    from google.colab import files; files.download(out)
print('wrote', out)

### Optional: run everything in one cell
```python
run('all')   # resumable; skips finished stages
```